# Python Data Engineering — Dataset Setup

Run this once to verify realistic files. The dataset files are supplied in the accompanying `data/` folder; no PostgreSQL and no external packages are needed.

> **Realistic Dataset v2:** Shared across Python, Pandas, and PySpark. Existing columns retained. Synthetic data includes intentional dirty records and malformed JSON for interview exercises.


In [ ]:
from pathlib import Path
import json, csv, re, collections, itertools, heapq, statistics, datetime, zoneinfo, math

DATA_DIR = Path('data')
if not DATA_DIR.exists():
    DATA_DIR = Path('python_de_practice/data')
assert DATA_DIR.exists(), 'Run the dataset setup notebook first, or put the data/ folder next to this notebook.'

def load(name):
    with (DATA_DIR / f'{name}.json').open(encoding='utf-8') as f:
        return json.load(f)

def read_jsonl(name):
    with (DATA_DIR / f'{name}.jsonl').open(encoding='utf-8') as f:
        for line in f:
            yield json.loads(line)
print('Data folder:', DATA_DIR.resolve())


In [ ]:
for name in ['customers','products','orders','order_items','events','transactions','pipeline_logs','cdc_changes']:
    print(f'{name:18}',len(load(name)))

All data is deterministic and pre-generated. `customers_dirty.json`, `events_duplicates.jsonl`, and `malformed_events.jsonl` contain intentional data-quality issues. Output files from exercises will be saved in `data/`.

In [ ]:
import csv
import json
from pathlib import Path

# Adjust this path if your data folder is elsewhere
DATA = Path("data")

csv_path = DATA / "customers.csv"
json_path = DATA / "customers.json"
jsonl_path = DATA / "customers.jsonl"

# Load existing customer records
with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    customers = list(reader)

# Select an existing customer with a valid email.
# Preserve existing blank and whitespace-only test cases.
target = next(
    c for c in customers
    if c.get("email") and c["email"].strip()
)

target_id = target["customer_id"]

# Set email to NULL
target["email"] = ""

# Update CSV
with csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(customers)

# Update JSON and JSONL, preserving existing field types
with json_path.open("r", encoding="utf-8") as f:
    customers_json = json.load(f)

for c in customers_json:
    if str(c["customer_id"]) == str(target_id):
        c["email"] = None

with json_path.open("w", encoding="utf-8") as f:
    json.dump(customers_json, f, indent=2, ensure_ascii=False)

with jsonl_path.open("w", encoding="utf-8") as f:
    for c in customers_json:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"Updated customer_id: {target_id}")
print("CSV, JSON and JSONL updated successfully.")